# Specter Mutual: fallback notebooks

> **Fallback only.** [the guide](https://rocketride-org.github.io/rocketride-workshops/modules/specter-mutual/) is the walkthrough: participants start the project by hand, paste the files that must be exact, ask Claude Code to run and build the rest, and deploy from the App Builder. These notebooks are for anyone who can't get a step running locally. Each guide step's **Stuck? Use the fallback** fold says which part to run.

**What the project builds:** an AI claims reviewer for Specter Mutual, a fictional insurer that covers paranormal damage. A RocketRide **agent** reads each claim and recommends approve, deny or escalate. A **policy engine**, plain code with no AI in it, holds that recommendation to the rules. A **claims portal** puts it in front of people.

| Part | Fallback for the guide's step | What it does |
| --- | --- | --- |
| [01-project-init.ipynb](01-project-init.ipynb) | Start the project (the manifest edit) | Connects, checks the app exists, writes `authenticated` and `include` into its manifest |
| [02-pipeline-build.ipynb](02-pipeline-build.ipynb) | Build the brain | Writes the test claims, both pipelines and `policy.py`, then runs the three beats and the fraud test |
| [03-ui-build.ipynb](03-ui-build.ipynb) | Build the portal | Writes the finished reference portal into `apps/specter-mutual/` |

> Before any part: do the guide's **Start the project** step (`rocketride init` and `rocketride app create` in your project folder). Save this notebook and the part you need into that folder, next to `.env`, and run `python3 -m pip install nbformat` (Windows: `python -m pip install nbformat`) once: each part loads this notebook with `%run`, which needs it. The notebooks never install, scaffold or deploy.

## Shared helpers

One RocketRide client, the paths every part uses, and two helpers. The client signs in with the `.env` that the guide's terminal commands wrote into your project folder, so open these notebooks from that folder. This cell doesn't touch the network, so `%run` can load it: each part connects with `await connect()` right after.

In [ ]:
# Shared: the client, paths and helpers for every part. Safe to re-run.
import json
from pathlib import Path
from rocketride import RocketRideClient

WORKSPACE = Path.cwd()                         # your project folder: the one you initialized
APP_ROOT = WORKSPACE / "apps" / "specter-mutual"
PIPELINES = WORKSPACE / "pipelines"

if not (WORKSPACE / ".env").exists():
    raise RuntimeError("No .env here. Open this notebook from your project folder, the one you "
                       "initialized (see the guide's Start the project step).")
if "client" not in globals():
    client = RocketRideClient()  # reads ROCKETRIDE_URI and ROCKETRIDE_APIKEY from .env

async def connect():
    """Connect the shared client once. Every part runs this right after loading this notebook."""
    if not client.is_connected():
        await client.connect()
    print("Connected to RocketRide.")

def replay(notebook):
    """Re-run a part's file-writing cells (%%writefile, or cells headed "# Write"). Skip-ahead cells use it."""
    nb = json.loads(Path(notebook).read_text(encoding="utf-8"))
    for cell in nb["cells"]:
        src = "".join(cell["source"])
        if cell["cell_type"] != "code":
            continue
        if src.startswith("%%writefile "):
            first, _, body = src.partition("\n")
            path = Path(first.split(None, 1)[1].strip())
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(body, encoding="utf-8", newline="\n")
            print("Wrote", path.as_posix())
        elif src.startswith("# Write"):
            exec(src, globals())

print("Shared helpers ready.")

## What's next

- **Docs:** [docs.rocketride.org](https://docs.rocketride.org)
- **Help and community:** [RocketRide Discord](https://discord.gg/PMXrtenMsY)
- **Back to the hub:** [RocketRide workshops](https://github.com/rocketride-org/rocketride-workshops)